In [2]:
import mysql.connector
from faker import Faker
import random
from datetime import datetime, timedelta

In [3]:
# ============================================================
# CONFIGURATION
# ============================================================

DB_HOST = "localhost"
DB_PORT = 3306
DB_USER = "student"
DB_PASSWORD = "student"
DB_NAME = "ecommerce"

CUSTOMERS_COUNT = 100_000
PRODUCTS_COUNT = 20_000
ORDERS_COUNT = 500_000
ORDER_ITEMS_COUNT = 2_000_000
PAYMENTS_COUNT = 500_000

BATCH_SIZE = 5_000

fake = Faker()

In [4]:
# ============================================================
# DATABASE CONNECTION
# ============================================================

connection = mysql.connector.connect(
    host=DB_HOST,
    port=DB_PORT,
    user=DB_USER,
    password=DB_PASSWORD
)

cursor = connection.cursor()

cursor.execute(f"""
CREATE DATABASE IF NOT EXISTS {DB_NAME}
"""

)

connection.database = DB_NAME

In [5]:
# ============================================================
# DROP TABLES
# ============================================================

cursor.execute("DROP TABLE IF EXISTS payments")
cursor.execute("DROP TABLE IF EXISTS order_items")
cursor.execute("DROP TABLE IF EXISTS orders")
cursor.execute("DROP TABLE IF EXISTS products")
cursor.execute("DROP TABLE IF EXISTS customers")

connection.commit()

In [6]:
# ============================================================
# CREATE CUSTOMERS
# ============================================================

print("Creating customers table...")

cursor.execute("""
CREATE TABLE customers (
    customer_id BIGINT PRIMARY KEY,
    first_name VARCHAR(100),
    last_name VARCHAR(100),
    email VARCHAR(255),
    country VARCHAR(100),
    city VARCHAR(100),
    signup_date DATE
)
""")


Creating customers table...


In [12]:
# ============================================================
# CREATE PRODUCTS
# ============================================================

print("Creating products table...")

cursor.execute("""
CREATE TABLE products (
    product_id BIGINT PRIMARY KEY,
    product_name VARCHAR(255),
    category VARCHAR(100),
    price DECIMAL(10,2),
    stock_quantity INT
)
""")

Creating products table...


In [7]:
# ============================================================
# CREATE ORDERS
# ============================================================

print("Creating orders table...")

cursor.execute("""
CREATE TABLE orders (
    order_id BIGINT PRIMARY KEY,
    customer_id BIGINT,
    order_date DATETIME,
    status VARCHAR(30),
    shipping_country VARCHAR(100)
)
""")


Creating orders table...


In [8]:
# ============================================================
# CREATE ORDER ITEMS
# ============================================================

print("Creating order_items table...")

cursor.execute("""
CREATE TABLE order_items (
    order_item_id BIGINT PRIMARY KEY,
    order_id BIGINT,
    product_id BIGINT,
    quantity INT,
    unit_price DECIMAL(10,2)
)
""")

Creating order_items table...


In [9]:
print("Creating payments table...")

cursor.execute("""
CREATE TABLE payments (
    payment_id BIGINT PRIMARY KEY,
    order_id BIGINT,
    payment_method VARCHAR(50),
    payment_status VARCHAR(30),
    payment_date DATETIME,
    amount DECIMAL(10,2)
)
""")

connection.commit()


Creating payments table...


In [10]:
# ============================================================
# 1. GENERATE CUSTOMERS
# ============================================================

print("Generating customers...")

countries = [
    "Egypt",
    "Saudi Arabia",
    "UAE",
    "Jordan",
    "Kuwait",
    "Qatar"
]

customers = []

for customer_id in range(1, CUSTOMERS_COUNT + 1):

    first_name = fake.first_name()
    last_name = fake.last_name()

    customers.append((
        customer_id,
        first_name,
        last_name,
        f"{first_name.lower()}.{last_name.lower()}{customer_id}@example.com",
        random.choice(countries),
        fake.city(),
        fake.date_between(
            start_date="-5y",
            end_date="today"
        )
    ))

    if len(customers) >= BATCH_SIZE:

        cursor.executemany("""
        INSERT INTO customers
        VALUES (%s,%s,%s,%s,%s,%s,%s)
        """, customers)

        connection.commit()

        customers = []

        print(f"Customers inserted: {customer_id:,}")

# Insert remaining
if customers:
    cursor.executemany("""
    INSERT INTO customers
    VALUES (%s,%s,%s,%s,%s,%s,%s)
    """, customers)

    connection.commit()

Generating customers...
Customers inserted: 5,000
Customers inserted: 10,000
Customers inserted: 15,000
Customers inserted: 20,000
Customers inserted: 25,000
Customers inserted: 30,000
Customers inserted: 35,000
Customers inserted: 40,000
Customers inserted: 45,000
Customers inserted: 50,000
Customers inserted: 55,000
Customers inserted: 60,000
Customers inserted: 65,000
Customers inserted: 70,000
Customers inserted: 75,000
Customers inserted: 80,000
Customers inserted: 85,000
Customers inserted: 90,000
Customers inserted: 95,000
Customers inserted: 100,000


In [13]:
# ============================================================
# 2. GENERATE PRODUCTS
# ============================================================

print("Generating products...")

categories = [
    "Electronics",
    "Clothing",
    "Shoes",
    "Home",
    "Beauty",
    "Sports",
    "Books",
    "Gaming",
    "Accessories",
    "Furniture"
]

products = []

for product_id in range(1, PRODUCTS_COUNT + 1):

    products.append((
        product_id,
        fake.catch_phrase(),
        random.choice(categories),
        round(random.uniform(5, 2000), 2),
        random.randint(0, 1000)
    ))

    if len(products) >= BATCH_SIZE:

        cursor.executemany("""
        INSERT INTO products
        VALUES (%s,%s,%s,%s,%s)
        """, products)

        connection.commit()

        products = []

        print(f"Products inserted: {product_id:,}")

if products:

    cursor.executemany("""
    INSERT INTO products
    VALUES (%s,%s,%s,%s,%s)
    """, products)

    connection.commit()


Generating products...
Products inserted: 5,000
Products inserted: 10,000
Products inserted: 15,000
Products inserted: 20,000


In [14]:
# ============================================================
# 3. GENERATE ORDERS
# ============================================================

print("Generating orders...")

statuses = [
    "completed",
    "completed",
    "completed",
    "pending",
    "cancelled",
    "shipped"
]

orders = []

start_date = datetime.now() - timedelta(days=730)

for order_id in range(1, ORDERS_COUNT + 1):

    customer_id = random.randint(1, CUSTOMERS_COUNT)

    order_date = start_date + timedelta(
        seconds=random.randint(
            0,
            730 * 24 * 60 * 60
        )
    )

    orders.append((
        order_id,
        customer_id,
        order_date,
        random.choice(statuses),
        random.choice(countries)
    ))

    if len(orders) >= BATCH_SIZE:

        cursor.executemany("""
        INSERT INTO orders
        VALUES (%s,%s,%s,%s,%s)
        """, orders)

        connection.commit()

        orders = []

        print(f"Orders inserted: {order_id:,}")

if orders:

    cursor.executemany("""
    INSERT INTO orders
    VALUES (%s,%s,%s,%s,%s)
    """, orders)

    connection.commit()

Generating orders...
Orders inserted: 5,000
Orders inserted: 10,000
Orders inserted: 15,000
Orders inserted: 20,000
Orders inserted: 25,000
Orders inserted: 30,000
Orders inserted: 35,000
Orders inserted: 40,000
Orders inserted: 45,000
Orders inserted: 50,000
Orders inserted: 55,000
Orders inserted: 60,000
Orders inserted: 65,000
Orders inserted: 70,000
Orders inserted: 75,000
Orders inserted: 80,000
Orders inserted: 85,000
Orders inserted: 90,000
Orders inserted: 95,000
Orders inserted: 100,000
Orders inserted: 105,000
Orders inserted: 110,000
Orders inserted: 115,000
Orders inserted: 120,000
Orders inserted: 125,000
Orders inserted: 130,000
Orders inserted: 135,000
Orders inserted: 140,000
Orders inserted: 145,000
Orders inserted: 150,000
Orders inserted: 155,000
Orders inserted: 160,000
Orders inserted: 165,000
Orders inserted: 170,000
Orders inserted: 175,000
Orders inserted: 180,000
Orders inserted: 185,000
Orders inserted: 190,000
Orders inserted: 195,000
Orders inserted: 200,000

In [15]:
# ============================================================
# 4. GENERATE ORDER ITEMS
# ============================================================

print("Generating order items...")

order_items = []

for order_item_id in range(1, ORDER_ITEMS_COUNT + 1):

    order_id = random.randint(1, ORDERS_COUNT)

    product_id = random.randint(1, PRODUCTS_COUNT)

    quantity = random.randint(1, 5)

    unit_price = round(
        random.uniform(5, 2000),
        2
    )

    order_items.append((
        order_item_id,
        order_id,
        product_id,
        quantity,
        unit_price
    ))

    if len(order_items) >= BATCH_SIZE:

        cursor.executemany("""
        INSERT INTO order_items
        VALUES (%s,%s,%s,%s,%s)
        """, order_items)

        connection.commit()

        order_items = []

        print(
            f"Order items inserted: "
            f"{order_item_id:,}"
        )

if order_items:

    cursor.executemany("""
    INSERT INTO order_items
    VALUES (%s,%s,%s,%s,%s)
    """, order_items)

    connection.commit()

Generating order items...
Order items inserted: 5,000
Order items inserted: 10,000
Order items inserted: 15,000
Order items inserted: 20,000
Order items inserted: 25,000
Order items inserted: 30,000
Order items inserted: 35,000
Order items inserted: 40,000
Order items inserted: 45,000
Order items inserted: 50,000
Order items inserted: 55,000
Order items inserted: 60,000
Order items inserted: 65,000
Order items inserted: 70,000
Order items inserted: 75,000
Order items inserted: 80,000
Order items inserted: 85,000
Order items inserted: 90,000
Order items inserted: 95,000
Order items inserted: 100,000
Order items inserted: 105,000
Order items inserted: 110,000
Order items inserted: 115,000
Order items inserted: 120,000
Order items inserted: 125,000
Order items inserted: 130,000
Order items inserted: 135,000
Order items inserted: 140,000
Order items inserted: 145,000
Order items inserted: 150,000
Order items inserted: 155,000
Order items inserted: 160,000
Order items inserted: 165,000
Orde

Order items inserted: 1,355,000
Order items inserted: 1,360,000
Order items inserted: 1,365,000
Order items inserted: 1,370,000
Order items inserted: 1,375,000
Order items inserted: 1,380,000
Order items inserted: 1,385,000
Order items inserted: 1,390,000
Order items inserted: 1,395,000
Order items inserted: 1,400,000
Order items inserted: 1,405,000
Order items inserted: 1,410,000
Order items inserted: 1,415,000
Order items inserted: 1,420,000
Order items inserted: 1,425,000
Order items inserted: 1,430,000
Order items inserted: 1,435,000
Order items inserted: 1,440,000
Order items inserted: 1,445,000
Order items inserted: 1,450,000
Order items inserted: 1,455,000
Order items inserted: 1,460,000
Order items inserted: 1,465,000
Order items inserted: 1,470,000
Order items inserted: 1,475,000
Order items inserted: 1,480,000
Order items inserted: 1,485,000
Order items inserted: 1,490,000
Order items inserted: 1,495,000
Order items inserted: 1,500,000
Order items inserted: 1,505,000
Order it

In [17]:
# ============================================================
# 5. GENERATE PAYMENTS
# ============================================================

print("Generating payments...")

payment_methods = [
    "credit_card",
    "debit_card",
    "cash",
    "paypal",
    "bank_transfer"
]

payment_statuses = [
    "paid",
    "paid",
    "paid",
    "failed",
    "refunded"
]

payments = []

for payment_id in range(1, PAYMENTS_COUNT + 1):

    order_id = payment_id

    payment_date = start_date + timedelta(
        seconds=random.randint(
            0,
            730 * 24 * 60 * 60
        )
    )

    payments.append((
        payment_id,
        order_id,
        random.choice(payment_methods),
        random.choice(payment_statuses),
        payment_date,
        round(random.uniform(10, 5000), 2)
    ))

    if len(payments) >= BATCH_SIZE:

        cursor.executemany("""
        INSERT INTO payments
        VALUES (%s,%s,%s,%s,%s,%s)
        """, payments)

        connection.commit()

        payments = []

        print(
            f"Payments inserted: "
            f"{payment_id:,}"
        )

if payments:

    cursor.executemany("""
    INSERT INTO payments
    VALUES (%s,%s,%s,%s,%s,%s)
    """, payments)

    connection.commit()

Generating payments...
Payments inserted: 5,000
Payments inserted: 10,000
Payments inserted: 15,000
Payments inserted: 20,000
Payments inserted: 25,000
Payments inserted: 30,000
Payments inserted: 35,000
Payments inserted: 40,000
Payments inserted: 45,000
Payments inserted: 50,000
Payments inserted: 55,000
Payments inserted: 60,000
Payments inserted: 65,000
Payments inserted: 70,000
Payments inserted: 75,000
Payments inserted: 80,000
Payments inserted: 85,000
Payments inserted: 90,000
Payments inserted: 95,000
Payments inserted: 100,000
Payments inserted: 105,000
Payments inserted: 110,000
Payments inserted: 115,000
Payments inserted: 120,000
Payments inserted: 125,000
Payments inserted: 130,000
Payments inserted: 135,000
Payments inserted: 140,000
Payments inserted: 145,000
Payments inserted: 150,000
Payments inserted: 155,000
Payments inserted: 160,000
Payments inserted: 165,000
Payments inserted: 170,000
Payments inserted: 175,000
Payments inserted: 180,000
Payments inserted: 185,00

In [18]:
# ============================================================
# CLOSE CONNECTION
# ============================================================

cursor.close()
connection.close()

print("====================================")
print("DATA GENERATION COMPLETED")
print("====================================")

DATA GENERATION COMPLETED
